In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from huggingface_hub import login

login("")

In [ ]:
import transformers
import torch
import os
from transformers import pipeline
from tqdm import tqdm
from datasets import load_dataset
import pandas as pd

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

CHECKPOINT_FILE = "checkpoint.txt"


def get_last_row():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            return int(f.read().strip())
    return 0


def save_last_row(idx):
    with open(CHECKPOINT_FILE, "w") as f:
        f.write(str(idx))


pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)


def generate_answer(problem):
    messages = [
        {
            "role": "user",
            "content": problem,
        }
    ]

    outputs = pipeline(
        messages,
        max_new_tokens=2048,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
    )
    return outputs[0]["generated_text"][-1]["content"]

In [ ]:
dataset = load_dataset("large-traversaal/mgsm_urdu_final", split="test")
df = dataset.to_pandas()
# results = []

# for row in tqdm(dataset):
#     response = generate_answer(row["urdu_question"])  # Your model response

#     results.append(
#         {
#             "question": row["urdu_question"],
#             "answer": row["answer_number"],
#             "response": response,
#         }
#     )

# df = pd.DataFrame(results)
# df.to_csv("llama-response.csv", index=False)

In [ ]:
start_idx = get_last_row()
print(f"Resuming from row {start_idx}")
for idx, row in tqdm(df.iloc[start_idx:].iterrows(), total=len(df) - start_idx):
    response = generate_answer(row["urdu_question"])
    print(f"Processed row {idx}")
    print(f"question: {row['urdu_question']}")
    print("\n\n")
    results = []
    results.append(
        {
            "row_idx": idx,
            "question": row["urdu_question"],
            "answer": row["answer_number"],
            "response": response,
        }
    )
    # Save immediately after successful processing
    save_last_row(idx + 1)
    scores_df = pd.DataFrame(
        results,
    )
    scores_df.to_csv(
        "llama-instruct-response.csv",
        mode="a",
        index=False,
        header=not os.path.exists("llama-instruct-response.csv"),
    )

In [ ]:
df = pd.read_csv("llama-instruct-response.csv")

validation_results = []

model_id = "openai/gpt-oss-20b"

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype="auto",
    device_map="auto",
)

for _, row in tqdm(df.iterrows()):

    prompt = f"""You are an answer validator for Urdu math reasoning problems.
    
    Input:
    
    * Question: {row['question']}
    * Solution: {row['response']}
    * Expected Answer: {row['answer']}
    
    Task:
    
    1. Read the Urdu math problem and the provided solution.
    2. Determine the final answer produced by the solution.
    3. Compare it with the Expected Answer.
    4. Treat numerically equivalent values as equal:
    
       * 2 = 2.0 = 2.00
       * 5 = 5.000
       * Ignore insignificant trailing zeros in decimals.
    5. If both answers represent the same numeric value, return:
       True
    6. Otherwise return:
       False
    
    Rules:
    
    * Return ONLY one word: True or False.
    * Do not provide explanations, reasoning, calculations, or extra text.
    * Comparison should be based on the final answer only.
    * For numeric answers, compare by value, not string format.
    * If the solution's final answer cannot be determined with confidence, return False.
     """

    messages = [{"role": "user", "content": prompt}]
    outputs = pipe(messages, max_new_tokens=1000)

    result = (
        True
        if "True.assistantfinalTrue" in outputs[0]["generated_text"][-1]["content"]
        else False
    )
    print(outputs[0]["generated_text"][-1]["content"])
    print(result)
    print("\n\n")
    validation_results.append(result)

df["is_correct"] = validation_results

df.to_csv("llama-instruct_correct.csv", index=False)

print(df.head())